In [19]:
#!/Users/ginoprasad/miniconda3/envs/google-api/bin/python3

import os, glob
import sys
import settings
import json
import concurrent.futures
from google.cloud import pubsub_v1
import google.auth
import subprocess as sp
import time
import base64
import re

import utils
from tqdm import tqdm

In [2]:
def extract_text(payload):
    if 'parts' in payload:
        return ''.join(map(extract_text, payload['parts']))
    elif payload.get('mimeType') == 'text/plain':
        data = payload.get('body', {}).get('data', '')
        if data:
            return base64.urlsafe_b64decode(data).decode('utf-8', errors='ignore')
    return ''


list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1000).execute()
messages = list_res.get('messages', [])

In [71]:
regex_include = [
    r'(?<=Get Code\r\n\[)https://.*?(?=[ \]])',
    r'(?<=Enter this code to sign in\r\n\r\n)[0-9][0-9][0-9][0-9]',
    r'(?<=Yes, This Was Me\r\n\[)https://.*?(?=[\]])',
]

regex_exclude = [
    'Please review who’s using your Netflix account',
    'We’ve updated your account with your new payment info',
    'Your Netflix Household has been confirmed',
    'we’re updating our prices',
    "Here's a quick summary of key updates to reflect our new features and services",
    'We recently announced that',
    'Your 4K upgrade ends soon'
]

emails = [
    'info@account.netflix.com'
]

In [ ]:
for message in tqdm(messages):
    eid = message['id']
    msg = utils.service.users().messages().get(
        userId='me', id=eid, format='full'
    ).execute()

    
    email_from = ''.join([x['value'] for x in msg['payload']['headers'] if x['name'] == 'From'])
    email_from = re.search(r'(?<=<).*(?=>)', email_from).group()
    
    labels = msg.get('labelIds', [])
    if 'INBOX' in labels and 'SENT' not in labels and email_from in emails:
        payload = extract_text(msg['payload'])
                
        for regex in regex_include:
            code = re.search(regex, payload)
            if code is not None:
                code = code.group()
                break
        
        if code is None:
            exclude = False
            for regex in regex_exclude:
                code = re.search(regex, payload)
                if code is not None:
                    exclude = True
                    break
            if not exclude:
                with open(settings.payload_path, 'w') as outfile:
                    outfile.write(payload)
            continue
        print(code)

  0%|                          | 2/500 [00:00<02:05,  3.97it/s]

1073


  1%|▏                         | 3/500 [00:00<02:16,  3.65it/s]

1073


  1%|▏                         | 4/500 [00:01<02:05,  3.96it/s]

...

 21%|████▉                   | 104/500 [00:31<02:01,  3.27it/s]

https://www.netflix.com/account/travel/verify?nftoken=BgixsOvcAxKZAcYv9og57XmpwX75CM1fwGyjs/tgehs0QcKMZDVFw4VsG4ku2aSpKBHCJ2atHaRTyRquMxsrUYqgaETrRR3o+jyhHrjbhf3ZmxaGOwAQKvm0s+i/6/Pq7yT8ASKllAEFqKhVK4zJAVlJrCpdRFoHBC67co8KL1gn2Q1V40V/Ql/Wp4M6A6GsPqESUyIZfUejHkyvyG6TvIDVOhgGIg4KDMMudt8WHsd7qoGO3g==&messageGuid=833aeb91-352e-42f5-ba55-cfc00db74917


 21%|█████                   | 106/500 [00:32<02:12,  2.98it/s]

https://www.netflix.com/account/update-primary-location?nftoken=BgixsOvcAxKcAdsukHCZdZywPGKxPC93tabgq8To2XAnVZUfuTylGFAe+BqTST8TD+0zv1AJZIAqrARLvOL69fqlFrBom+bQbYc3GFk36wl6tjU8ed2uEBvVfiaoCcpO/l7AsXJF7CmyGWW2f5akBExxjaHOzUxdRV+iyWk1RZ14mUCqH74MdHT56U+b+U1HZGG8SX4FK87uyeBAHHX38PZeKKL0RhgGIg4KDOx9cVDETQMR59grvw==&g=dbd45b61-40e9-4ee8-ba52-4d74cf5b531c&lnktrk=EVO&operation=update&lkid=UPDATE_HOUSEHOLD_REQUESTED_OTP_CTA


 72%|█████████████████▍      | 362/500 [01:46<00:37,  3.68it/s]